# 21장 · 열어서 검증받기: 읽지 않고 확인하기

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 설치는 없고, 구글 계정만 있으면 됩니다.

**읽는 법.** 흰 바탕의 글(지금 이것)은 설명이라 실행하지 않습니다. **회색 상자**만 코드이고, 왼쪽의 **▶** 또는 `Shift`+`Enter` 로 실행합니다.

**순서.** 번호 차례대로 끝까지 갑니다. **1 준비**부터 시작해 마지막 번호까지 위에서 아래로 내려가면 됩니다. ⚠ 가운데부터 누르면 앞에서 만든 것이 없어 오류가 납니다.

📖 본문 학습 페이지: [21장 · 열어서 검증받기: 읽지 않고 확인하기](https://grow.minds.kr/textbooks/css-methods/causal/book/ch21-열어서-검증받기.html)

## 1. 준비

아래 **회색 상자 둘**을 차례로 실행하세요. 둘 다 해야 그다음이 돌아갑니다.

1. 첫째 = 자료와 코드를 내려받습니다. **몇 초 걸리고**, 「경고」 문구가 떠도 정상입니다. 상자 아래에 `/content/css-methods-causal-code` 가 찍히면 성공입니다.
2. 둘째 = 도구와 도우미 함수를 불러옵니다. **「준비 끝」**이 찍히면 됩니다.

---

### ⛔ 둘째 상자의 코드는 지금 이해하지 않아도 됩니다

상자가 길어 놀랄 수 있습니다. 이 책 전체가 쓰는 **도우미 넷**을 미리 만들어 두는 곳이라 그렇습니다. 지금은 **이름과 하는 일만** 훑고 넘어가세요. 안에 든 식은 필요한 장에서 하나씩 만납니다.

| 이름 | 하는 일 | 처음 쓰는 곳 |
|---|---|---|
| `load` | 자료 한 벌을 불러오고 주의 점검 실패자를 걸러 낸다 | 2장 |
| `cronbach` | 척도 신뢰도(알파) | 4장 |
| `cohen_d` | 두 집단 차이의 효과크기 | 13장 |
| `ols` | 회귀의 계수·표준오차·p·R² | 14장 |

⚠ `np.linalg.lstsq` 처럼 낯선 이름이 보여도 괜찮습니다. **그것이 이 책이 「계산은 AI가」라고 말하는 곳입니다.**

⭐ **그리고 이 넷은 여러분이 검증할 대상이 아닙니다.** 이 책이 검증해 둔 도구입니다(배포 때마다 대조 배터리가 이 함수들의 출력을 본문 수치와 맞춰 봅니다). 여러분이 읽어야 할 코드는 **AI가 준 코드**이고, 그 훈련은 3장부터 시작합니다.

📖 이 넷을 다시 보고 싶으면 책 부록 E.1b 에 같은 코드가 있습니다.

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. AI 가 준 코드. 읽지 않는다
함수 하나로 취급하고 밖에서 조건만 겁니다. 먼저 출력만 보고 구별할 수 있는지 확인하세요.

In [ ]:
def ai_회귀(y, x):
    X = np.column_stack([np.asarray(x, float)])
    b, *_ = np.linalg.lstsq(X, np.asarray(y, float), rcond=None)
    resid = np.asarray(y, float) - X @ b
    n, k = X.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X.T @ X)))
    return float(b[-1]), float(se[-1])
svy = load("svy")
x, y = svy.hjs.values, svy.mil.values
print(round(ai_회귀(y, x)[0], 3), round(ai_회귀(y, x)[1], 3), "|",
      round(float(ols(y, [x])[0][1]), 3), round(float(ols(y, [x])[1][1]), 3))
# 0.976 0.008 | 0.978 0.051
# 기울기가 소수 둘째 자리까지 같다. R2 도 둘 다 .395 다. 출력으로는 못 가른다.

## 3. ⭐⭐ 불변식을 건다
답이 얼마인지는 몰라도, **입력을 이렇게 바꾸면 출력이 저렇게 바뀌어야 한다**는 것은 압니다.

In [ ]:
def ai_기울기(yy, xx):    return ai_회귀(yy, xx)[0]
def 바른_기울기(yy, xx):                 # SE 없이 기울기만 (특이행렬 회피)
    yy = np.asarray(yy, float)
    X = np.column_stack([np.ones(len(yy)), np.asarray(xx, float)])
    return float(np.linalg.lstsq(X, yy, rcond=None)[0][1])
g = np.random.default_rng(73)
i = g.permutation(len(x))
for 이름, f in (("AI 판 ", ai_기울기), ("바른 판", 바른_기울기)):
    b0 = f(y, x)
    print(이름, "① 뒤섞기", round(f(y[i], x[i]) - b0, 6),
                "③ Y+5", round(f(y + 5, x) - b0, 4),
                "④ Y 상수", round(f(np.full(len(y), 4.0), x), 4))
# AI 판  ① 0.0 ③ 0.9729 ④ 0.7783   ← ③④ 에서 죽는다
# 바른 판 ① 0.0 ③ -0.0 ④ -0.0
# 코드는 한 줄도 안 읽었다. 원인(절편 누락)은 알아내도 되고 안 알아내도 된다.

## 4. ⭐ 시험을 시험한다
①②만 가지고 있었다면 위 코드를 통과시켰습니다. 고장을 일부러 심어 **어느 시험이 무엇을 잡는지** 보세요.

In [ ]:
def 고장B(yy, xx): return 바른_기울기(xx, yy)          # X·Y 뒤바꿈
def 고장C(yy, xx):                                    # 앞 절반만
    h = len(yy) // 2
    return 바른_기울기(np.asarray(yy)[:h], np.asarray(xx)[:h])
def 격자(f):
    b0 = f(y, x); gg = np.random.default_rng(73)
    널 = [f(y, gg.permutation(x)) for _ in range(200)]
    return ["통과" if t else "실패" for t in (
        abs(f(y[i], x[i]) - b0) < 1e-6,
        abs(f(y, x * 10) * 10 - b0) < 1e-4,
        abs(f(y + 5, x) - b0) < 1e-4,
        abs(f(np.full(len(y), 4.0), x)) < 1e-4,
        abs(f(np.r_[y, y], np.r_[x, x]) - b0) < 1e-4,
        abs(float(np.mean(널))) < .05)]
for nm, f in (("바른 판", 바른_기울기), ("A 절편없음", ai_기울기),
              ("B X·Y뒤바꿈", 고장B), ("C 앞절반", 고장C)):
    print(f"{nm:11s}", round(f(y, x), 3), 격자(f))
# 어느 시험도 고장 셋을 다 못 잡는다. 고장마다 잡는 시험이 다르다.
# 그러므로 시험 묶음은 개수가 아니라 다양성으로 산다.

## 5. 익명 공개 점검
심사에 낼 폴더에서 신원이 새는 곳을 훑습니다. 이름·기관은 자기 것으로 바꾸세요.

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "check_anonymity.py", ".",
                      "--name", "홍길동", "--org", "제주대"],
                     capture_output=True, text=True, encoding="utf-8").stdout[:1200])
# 걸린 것이 있으면 확실히 익명이 아니다. 통과가 익명을 증명하지는 않는다.

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.